# Fraud Prediction Consumer

This notebook listens to the Kafka topic:

fraud-predictions

Every prediction published by Notebook 18 is displayed in real time.

This notebook represents downstream systems such as dashboards, databases and alert services.

## Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("FraudPredictionConsumer")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

## Stop Existing Streams

In [ ]:
for stream in spark.streams.active:
    stream.stop()

print("Stopped existing streams.")

## Read Kafka Topic

In [ ]:
stream_df = (

    spark.readStream

    .format("kafka")

    .option(
        "kafka.bootstrap.servers",
        "localhost:9092"
    )

    .option(
        "subscribe",
        "fraud-predictions"
    )

    .option(
        "startingOffsets",
        "latest"
    )

    .load()

)

## Prediction Schema

In [ ]:
schema = StructType([

    StructField("transaction_id", StringType()),

    StructField("event_timestamp", StringType()),

    StructField("Amount", DoubleType()),

    StructField("Prediction", IntegerType()),

    StructField("Fraud_Probability", DoubleType()),

    StructField("Risk_Level", StringType()),

    StructField("Status", StringType())

])

## Parse JSON

In [ ]:
parsed_df = (

    stream_df

    .selectExpr("CAST(value AS STRING)")

    .select(

        from_json(
            col("value"),
            schema
        ).alias("prediction")

    )

    .select("prediction.*")

)

## Verify Schema

In [ ]:
parsed_df.printSchema()

## Prediction Consumer Function

In [ ]:
def display_predictions(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return

    print("\n")
    print("=" * 120)
    print(f"Prediction Batch {batch_id}")
    print("=" * 120)

    print(
        pdf[
            [
                "transaction_id",
                "Amount",
                "Fraud_Probability",
                "Risk_Level",
                "Status"
            ]
        ]
    )

    frauds = pdf[pdf["Prediction"] == 1]

    if len(frauds):

        print("\n")
        print("🚨 FRAUD ALERT 🚨")
        print("=" * 120)

        print(
            frauds[
                [
                    "transaction_id",
                    "Amount",
                    "Fraud_Probability",
                    "Risk_Level"
                ]
            ]
        )

    print("\nReceived", len(pdf), "prediction(s)")

## Start Consumer

In [ ]:
query = (

    parsed_df

    .writeStream

    .foreachBatch(display_predictions)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../checkpoints/prediction_consumer"
    )

    .start()

)

## Keep Consumer Alive

In [ ]:
query.awaitTermination()